# Ejercicios Prácticos (Anexo) - Unidad 3, Sesión 2


## Configuración Previa


In [1]:
from google.colab import userdata
import os

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [2]:
!pip install -q grandalf
!pip install -q langgraph langchain-openai python-dotenv
from dotenv import load_dotenv
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 18.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
modelo = ChatOpenAI(
 model="google/gemma-4-31b-it:free",
 temperature=0,
 openai_api_key=os.getenv("OPENROUTER_API_KEY"),
 openai_api_base="https://openrouter.ai/api/v1",
)


## Ejercicio: Construyendo un Asistente con LangGraph


### Etapa 1: Chatbot Mínimo
Implementa el chatbot más sencillo posible: un grafo con un único nodo que llama al modelo.


In [ ]:
load_dotenv()
# Modelo (sustituir por OpenRouter si no se tiene OpenAI)
modelo = ChatOpenAI(
 model="google/gemma-4-31b-it:free",
 temperature=0,
 openai_api_key=os.getenv("OPENROUTER_API_KEY"),
 openai_api_base="https://openrouter.ai/api/v1",
)

# TODO 1: Define un nodo llamado `llamar_modelo` que:
# - reciba `state: MessagesState`
# - llame a modelo.invoke(state["messages"])
# - devuelva un dict con la clave "messages" conteniendo la respuesta

def llamar_modelo(state: MessagesState) -> dict:
    respuesta = modelo.invoke(state["messages"])
    return {"messages": [respuesta]}


# TODO 2: Construye el grafo
# - crea un StateGraph(MessagesState)
# - añade el nodo "chatbot" con la función llamar_modelo
# - conecta START → "chatbot" → END

builder = StateGraph(MessagesState)
builder.add_node("chatbot", llamar_modelo)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)


# TODO 3: Compila el grafo
graph = builder.compile()

# TODO 4: Visualiza el grafo en ASCII
print(graph.get_graph().draw_ascii())

# TODO 5: Invoca el grafo con la pregunta "¿Qué es Python?"
resultado = graph.invoke({
 "messages": [{"role": "user", "content": "¿Qué es Python?"}]
})

# Imprimir la respuesta del modelo (último mensaje del historial)
print(resultado["messages"][-1].content)

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | chatbot |   
 +---------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   
**Python** es un lenguaje de programación de **alto nivel, interpretado y de propósito general**. Fue creado por Guido van Rossum y lanzado por primera vez en 1991.

Para entenderlo mejor, aquí te explico sus características principales de forma sencilla:

### 1. ¿Qué significan esos términos técnicos?
*   **De alto nivel:** Significa que su sintaxis es muy parecida al lenguaje humano (específicamente al inglés), lo que lo hace mucho más fácil de leer y escribir que otros lenguajes como C++ o Java.
*   **Interpretado:** El código no necesita ser "compilado" (convertido a lenguaje máquina) antes de ejecutarse. Un programa llamado *intérprete* lee el código línea por línea y lo ejecuta al momento.
*   **De propósito general:** No está diseñado para una

#### Preguntas de Reflexión

1. **¿Cuántas líneas de código requiere este chatbot frente a `prompt | modelo` con LCEL?**

Este chatbot requiere más líneas de código que una cadena LCEL con `prompt | modelo`. En LangGraph hay que crear el grafo, añadir el nodo, conectar START con chatbot y este con END, compilarlo e invocarlo. LCEL es más breve y directo para cadenas simples como esta, pero LangGraph es más flexible cuando se necesitan flujos con estado, varios nodos, ramificaciones o ciclos.

2. ¿Qué tipo tiene el último elemento de resultado["messages"]? Imprime
type(resultado["messages"][-1]).
* Me devuelve AIMessage, que es la clase que LangChain usa para representar una respuesta del modelo.

In [ ]:
print(type(resultado["messages"][-1]))

<class 'langchain_core.messages.ai.AIMessage'>


3. ¿Qué pasaría si el nodo devolviera {"messages": resultado_completo} en vez de
{"messages": [respuesta]}? Pruébalo.

*  Al probar el nodo devolviendo {"messages": resultado_completo}, el grafo siguió funcionando y respondió correctamente.


In [ ]:
def llamar_modelo2(state: MessagesState) -> dict:
    resultado_completo = modelo.invoke(state["messages"])
    return {"messages": resultado_completo}


builder2 = StateGraph(MessagesState)

builder2.add_node("chatbot", llamar_modelo2)
builder2.add_edge(START, "chatbot")
builder2.add_edge("chatbot", END)

graph2 = builder2.compile()

resultado2 = graph2.invoke({
    "messages": [
        {"role": "user", "content": "¿Qué es Python?"}
    ]
})

print(resultado2["messages"][-1].content)

**Python** es un lenguaje de programación de **alto nivel, interpretado y de propósito general**. Fue creado por Guido van Rossum y lanzado por primera vez en 1991.

Para entenderlo mejor, aquí te explico sus características principales de forma sencilla:

### 1. ¿Qué significan esos términos técnicos?
*   **De alto nivel:** Significa que su sintaxis es muy parecida al lenguaje humano (específicamente al inglés), lo que lo hace mucho más fácil de leer y escribir que otros lenguajes como C++ o Java.
*   **Interpretado:** El código no necesita ser "compilado" (convertido a lenguaje máquina) antes de ejecutarse. Un programa llamado *intérprete* lee el código línea por línea y lo ejecuta al momento.
*   **De propósito general:** No está diseñado para una sola cosa. Puedes usarlo para crear páginas web, analizar datos, automatizar tareas, crear inteligencia artificial, etc.

### 2. Características principales
*   **Sintaxis limpia:** Python utiliza la **indentación** (los espacios al princi

### Etapa 2: Añadir Memoria con MemorySaver
Ahora vamos a darle memoria al chatbot sin escribir lógica de gestión de historial, simplemente


In [ ]:
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI

load_dotenv()
# Modelo (sustituir por OpenRouter si no se tiene OpenAI)
modelo = ChatOpenAI(
 model="google/gemma-4-31b-it:free",
 temperature=0,
 openai_api_key=os.getenv("OPENROUTER_API_KEY"),
 openai_api_base="https://openrouter.ai/api/v1",
)

def llamar_modelo(state: MessagesState) -> dict:
 return {"messages": [modelo.invoke(state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot", llamar_modelo)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# TODO 1: Crea un MemorySaver y compila el grafo CON ese checkpointer
checkpointer = MemorySaver()

graph = builder.compile(checkpointer=checkpointer)

# TODO 2: Define la configuración con un thread_id (puede ser cualquier string)
config = {"configurable": {"thread_id": "conversacion1"}}

# TODO 3: Bucle interactivo de conversación
# - leer input del usuario con input("Tú: ")
# - si escribe "salir", terminar
# - en otro caso, invocar graph con config y mostrar la respuesta

print("Chatbot con memoria. Escribe 'salir' para terminar.\n")
while True:
    user_msg = input("Tú: ").strip()
    if user_msg.lower() == "salir":
        break
    if not user_msg:
        continue
    resultado = graph.invoke(
        {"messages": [{"role": "user", "content": user_msg}]},
        config=config,
    )

    print(f"Bot: {resultado['messages'][-1].content}\n")

# TODO 4 (opcional): después de salir del bucle, imprime cuántos mensajes
# tiene el historial en el thread, usando graph.get_state(config)
estado = graph.get_state(config)

print(f"\nMensajes guardados en el thread:{len(estado.values['messages'])}")

Chatbot con memoria. Escribe 'salir' para terminar.

Tú: Hola, me llamo Yago
Bot: ¡Hola, Yago! Mucho gusto. ¿En qué puedo ayudarte hoy?

Tú: Estoy aprendiendo LangGraph
Bot: ¡Qué genial! **LangGraph** es actualmente una de las herramientas más potentes para construir agentes de IA, porque permite ir más allá de las simples cadenas (chains) y crear flujos de trabajo **cíclicos**.

Si vienes de usar LangChain, sabrás que las cadenas son lineales. LangGraph introduce el concepto de **Grafos**, lo que permite que el agente pueda:
1. **Iterar:** Intentar una tarea, revisar el resultado y, si no es correcto, volver atrás y corregirlo.
2. **Mantener Estado (State):** Recordar información a lo largo de todo el ciclo de vida del grafo.
3. **Control Humano (Human-in-the-loop):** Pausar la ejecución para que un humano apruebe una acción antes de que el agente la ejecute.

Para ayudarte mejor, ¿en qué punto te encuentras?

*   **¿Estás empezando desde cero?** (Puedo explicarte los conceptos de `St

RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-4-31b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False}}, 'user_id': 'user_3CinYn5E7gvVmxVuMT6jMgdjrum'}

#### Verificación: Conversación de Prueba

Mantén esta conversación y comprueba que el bot recuerda:
1. Tú: Hola, me llamo [tu nombre]
2. Tú: Estoy aprendiendo LangGraph
3. Tú: ¿Cómo me llamo? ← debe responder con tu nombre
4. Tú: ¿Sobre qué tema dije que estoy aprendiendo? ← debe decir LangGraph

#### Reto Adicional


1. Cambia el thread_id a otro valor distinto y reinicia la conversación. ¿El bot recuerda lo anterior?
Justifica por qué.

* Si cambio el thread_id a otro valor distinto, el bot no recuerda lo anterior.

* Esto ocurre porque MemorySaver guarda el estado de la conversación en la memoria RAM y lo identifica por el thread_id. Si uso un thread_id diferente, LangGraph no recupera la información del thread_id anterior.

* Además, al reiniciar el script, la memoria almacenada por se pierde, ya que no es persistente. Con MemorySaver la conversación solo se conserva mientras el proceso siga ejecutándose y mientras se use el mismo thread_id.

2. Reemplaza MemorySaver() por SqliteSaver.from_conn_string("chat.db") (necesita pip
install langgraph-checkpoint-sqlite). Reinicia el script entero y comprueba si el historial
sobrevive entre ejecuciones.

* El historial sobrevive entre ejecuciones. Al usar SqliteSaver, se guarda el estado de la conversación de forma persistente en el archivo `chat.db`, en lugar de mantenerlo únicamente en memoria RAM.

##### Código con SqliteSaver


In [ ]:
!pip install -q langgraph-checkpoint langgraph-checkpoint-sqlite
!pip install -U -q langgraph langgraph-checkpoint langgraph-checkpoint-sqlite

import os
from dotenv import load_dotenv

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_openai import ChatOpenAI

load_dotenv()

modelo = ChatOpenAI(
 model="google/gemma-4-31b-it:free",
 temperature=0,
 openai_api_key=os.getenv("OPENROUTER_API_KEY"),
 openai_api_base="https://openrouter.ai/api/v1",
)

def llamar_modelo(state: MessagesState) -> dict:
    return {"messages": [modelo.invoke(state["messages"])]}

builder = StateGraph(MessagesState)

builder.add_node("chatbot", llamar_modelo)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# Checkpointer persistente en SQLite
with SqliteSaver.from_conn_string("checkpoints.db") as checkpointer:
    graph = builder.compile(checkpointer=checkpointer)

    # Mantén el mismo thread_id entre ejecuciones para recuperar el historial
    config = {"configurable": {"thread_id": "conversacion_1"}}

    print("Chatbot con memoria SQLite. Escribe 'salir' para terminar.\n")

    while True:
        user_msg = input("Tú: ").strip()

        if user_msg.lower() == "salir":
            break

        if not user_msg:
            continue

        resultado = graph.invoke(
            {"messages": [{"role": "user", "content": user_msg}]},
            config=config,
        )

        print(f"Bot: {resultado['messages'][-1].content}\n")

    estado = graph.get_state(config)
    print(f"\nMensajes guardados en el thread: {len(estado.values['messages'])}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 1.2.15 requires langgraph<1.2.0,>=1.1.5, but you have langgraph 1.2.0 which is incompatible.
Chatbot con memoria SQLite. Escribe 'salir' para terminar.

Tú: Hola, me llamo Yago


RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-4-31b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False}}, 'user_id': 'user_3CinYn5E7gvVmxVuMT6jMgdjrum'}

#### Preguntas de Reflexión

1. ¿Cuánto código de gestión de historial has escrito tú? Compara con el chatbot manual del Ejercicio
3 de ejercicios.md.
* En este ejercicio he escrito muy poco código de gestión manual del historial. La gestión de memoria la realiza prácticamente entera LangGraph mediante MessagesState, MemorySaver y el thread_id.


2. ¿Qué representan los thread_id en una aplicación real (ej. WhatsApp, Slack, web)?
* Los thread_id serían el identificador único de una conversación.

3. ¿Cuándo elegirías MemorySaver vs SqliteSaver vs PostgresSaver?
* MemorySaver para pruebas porque guarda el historial en memoria RAM y no es persistente.
* SqliteSaver para proyectos pequeños o medianos donde quiera persistencia sin montar una infraestructura complicada.
* PostgresSaver para aplicaciones reales en producción. Es más complicado de complicado de configurar pero es mucho más robusto.

### Etapa 3: Enrutado Condicional - Asistente con Especialidades
Vamos a construir un asistente que clasifica la pregunta del usuario y la enruta a un experto distinto
según el tema.



In [4]:
import os
from typing import Annotated, TypedDict, Literal
from dotenv import load_dotenv

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

load_dotenv()

modelo = ChatOpenAI(
    model="google/gemma-4-31b-it:free",
    temperature=0,
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
)

# TODO 1: Define un TypedDict EstadoRouter con:
# - messages: Annotated[list, add_messages]
# - categoria: str
class EstadoRouter(TypedDict):
    messages: Annotated[list, add_messages]
    categoria: str


# --- Nodo clasificador ---
def clasificar(state: EstadoRouter) -> dict:
    pregunta = state["messages"][-1].content
    pregunta_lower = pregunta.lower()

    # Si la pregunta contiene "traduce" o "translate", se fuerza la categoría traduccion
    if "traduce" in pregunta_lower or "translate" in pregunta_lower:
        return {"categoria": "traduccion"}

    prompt = (
        "Clasifica la siguiente pregunta en UNA SOLA palabra entre: "
        "'python', 'sql', 'general' o 'traduccion'. Responde SOLO con esa palabra.\n\n"
        f"Pregunta: {pregunta}"
    )

    salida = modelo.invoke(prompt).content.strip().lower()

    # Normalizar la salida (puede contener basura)
    if "python" in salida:
        cat = "python"
    elif "sql" in salida:
        cat = "sql"
    elif "traduccion" in salida or "traducción" in salida:
        cat = "traduccion"
    else:
        cat = "general"

    return {"categoria": cat}


# --- TODO 2: Define tres nodos especialistas ---
# Cada uno recibe el estado, llama al modelo con un system prompt diferente
# y devuelve {"messages": [respuesta]}.
def experto_python(state: EstadoRouter) -> dict:
    system = "Eres un experto en Python. Responde con código limpio y comentarios."
    pregunta = state["messages"][-1].content

    respuesta = modelo.invoke([
        {"role": "system", "content": system},
        {"role": "user", "content": pregunta},
    ])

    return {"messages": [respuesta]}


def experto_sql(state: EstadoRouter) -> dict:
    system = (
        "Eres un experto en SQL. Responde con consultas claras, buenas prácticas "
        "y explicaciones breves cuando sea necesario."
    )
    pregunta = state["messages"][-1].content

    respuesta = modelo.invoke([
        {"role": "system", "content": system},
        {"role": "user", "content": pregunta},
    ])

    return {"messages": [respuesta]}


def experto_general(state: EstadoRouter) -> dict:
    system = (
        "Eres un asistente generalista. Responde de forma clara, útil y sencilla."
    )
    pregunta = state["messages"][-1].content

    respuesta = modelo.invoke([
        {"role": "system", "content": system},
        {"role": "user", "content": pregunta},
    ])

    return {"messages": [respuesta]}


# RETO ADICIONAL: experto_traductor
def experto_traduccion(state: EstadoRouter) -> dict:
    system = (
        "Eres un experto en traducción de textos. "
        "Traduce el texto del usuario de forma natural, precisa y correcta. "
        "Si el usuario indica un idioma de destino, traduce a ese idioma. "
        "Si no indica idioma de destino, traduce al español. "
        "Devuelve únicamente la traducción, sin explicaciones adicionales."
    )
    pregunta = state["messages"][-1].content

    respuesta = modelo.invoke([
        {"role": "system", "content": system},
        {"role": "user", "content": pregunta},
    ])

    return {"messages": [respuesta]}


# --- TODO 3: Define la función router ---
# Recibe el estado y devuelve el nombre del nodo siguiente
# según state["categoria"].
def decidir_ruta(
    state: EstadoRouter,
) -> Literal["experto_python", "experto_sql", "experto_general", "experto_traduccion"]:
    categoria = state["categoria"]

    if categoria == "python":
        return "experto_python"
    elif categoria == "sql":
        return "experto_sql"
    elif categoria == "traduccion":
        return "experto_traduccion"
    else:
        return "experto_general"


# --- Construir el grafo ---
builder = StateGraph(EstadoRouter)

builder.add_node("clasificar", clasificar)
builder.add_node("experto_python", experto_python)
builder.add_node("experto_sql", experto_sql)
builder.add_node("experto_general", experto_general)

builder.add_node("experto_traduccion", experto_traduccion)


builder.add_edge(START, "clasificar")

# TODO 4: Añade el edge condicional desde "clasificar" usando decidir_ruta
builder.add_conditional_edges("clasificar", decidir_ruta)

# TODO 5: Conecta cada experto a END
builder.add_edge("experto_python", END)
builder.add_edge("experto_sql", END)
builder.add_edge("experto_general", END)

builder.add_edge("experto_traduccion", END)


graph = builder.compile()

# Visualizar el grafo
print(graph.get_graph().draw_ascii())

# --- Probar con tres preguntas distintas ---
preguntas = [
    "¿Cómo escribo una list comprehension?",
    "¿Cómo hago un LEFT JOIN entre dos tablas?",
    "¿Cuál es la capital de Francia?",
    "Traduce al inglés: Estoy aprendiendo LangGraph.",
    "Translate the following text: I am learning artificial intelligence.",
]

for p in preguntas:
    print("\n" + "=" * 60)
    print(f"Pregunta: {p}")

    resultado = graph.invoke({
        "messages": [{"role": "user", "content": p}]
    })

    print(f"Categoría detectada: {resultado['categoria']}")
    # He tenido que reemplazar esta línea porque no me mostraba la respuesta
    # print(f"Respuesta: {resultado['messages'][-1].content[:200]}...")
    print(f"Respuesta: {resultado['messages'][-1].content}")

                                +-----------+                              
                                | __start__ |                              
                                +-----------+                              
                                      *                                    
                                      *                                    
                                      *                                    
                               +------------+                              
                               | clasificar |.                             
                          .....+------------+ .....                        
                      ....            .            ....                    
                 .....                 .               .....               
              ...                      .                    ...            
+-----------------+           +----------------+           +-------------+ 
| experto_ge

RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-4-31b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False}}, 'user_id': 'user_3CinYn5E7gvVmxVuMT6jMgdjrum'}

#### Verificación: Conversación de Prueba

Al ejecutar deberías obtener:
1. La pregunta de Python clasificada como python y respondida con código.
2. La de SQL clasificada como sql y respondida con una sentencia SELECT.
3. La de geografía clasificada como general y respondida sin código.

#### Reto Adicional


Añade un cuarto experto, experto_traduccion, que se active si la pregunta contiene la palabra
"traduce" o "translate". Modifica clasificar y decidir_ruta para soportarlo.


Se han añadido las líneas de código necesarias al código original.

#### Preguntas de Reflexión

##### 1. ¿Por qué clasificar devuelve {"categoria": cat} y no {"messages": [...]}? ¿Qué cambia en el estado tras ese nodo?


Porque el nodo clasificar no generan una respuesta para el usuario, sino que actualiza la variable interna de la categoría de la pregunta.

##### 2. ¿Sería posible hacer este enrutado con una Chain LCEL (prompt | modelo | parser)? ¿Qué problema encuentras?


Sí, sería posible usar una Chain LCEL para clasificar la pregunta, por ejemplo:

`chain = prompt | modelo | parser`

Esa chain podría devolver "python", "sql", "general" o "traduccion" pero después hay que decidir dinámicamente si se llama al experto de Python, al experto de SQL, al experto general o al experto de traducción.

Para hacerlo habría que añadir lógica externa con if/elif/else.

#####  3. ¿Qué ocurre si el clasificador devuelve una categoría inesperada (ej. "ruby")? ¿Cómo blindarías el router?

En este código eso no debería pasar, porque el clasificador normaliza la salida. Si no detecta que la categoría sea "python", "sql" o "traduccion", asigna por defecto la categoría "general".

Además, aunque por algún motivo llegara al router una categoría inesperada, como "ruby", el router también cuenta con un else final. Si la categoría no coincide con "python", "sql" o "traduccion", la ejecución se enviará directamente a "experto_general".

### Etapa 4: Agente ReAct con Tool Personalizada
Vamos a saltar al patrón estrella de LangGraph: un agente que decide cuándo usar herramientas. Este es
un anticipo de la Unidad 4.



In [5]:
import os
import math
from datetime import datetime
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

# --- TODO 1: Define 3 tools con el decorador @tool ---
# Cada tool DEBE tener un docstring claro: el LLM lo usará para decidir
# cuándo invocarla.
@tool
def calcular(expresion: str) -> str:
    """Evalúa una expresión matemática segura (suma, resta, mult, división,
    potencias, funciones de math). Ejemplo: 'sqrt(144) + 2**3'."""
    try:
        # eval restringido: solo names del módulo math + operadores
        permitidos = {
            k: getattr(math, k)
            for k in dir(math)
            if not k.startswith("_")
        }
        return str(eval(expresion, {"__builtins__": {}}, permitidos))
    except Exception as e:
        return f"Error: {e}"


# TODO: completa la siguiente tool. Debe devolver la fecha y hora actuales
# como string en formato "YYYY-MM-DD HH:MM".
@tool
def hora_actual() -> str:
    """Devuelve la fecha y hora actuales del sistema."""
    return datetime.now().strftime("%Y-%m-%d %H:%M")


# TODO: completa esta tool. Recibe un string `ciudad` y devuelve un string
# con el clima simulado (puedes hardcodear un dict de 3-4 ciudades).
@tool
def clima(ciudad: str) -> str:
    """Devuelve el clima actual de una ciudad española (datos simulados)."""
    datos_clima = {
        "madrid": "En Madrid hace sol, 18°C y viento suave.",
        "bilbao": "En Bilbao está nublado, 14°C y hay probabilidad de lluvia.",
        "barcelona": "En Barcelona hay intervalos nubosos, 20°C y humedad moderada.",
        "sevilla": "En Sevilla hace sol, 24°C y el ambiente es seco.",
    }

    ciudad_normalizada = ciudad.strip().lower()

    return datos_clima.get(
        ciudad_normalizada,
        f"No tengo datos simulados para {ciudad}. Prueba con Madrid, Bilbao, Barcelona o Sevilla."
    )


tools = [calcular, hora_actual, clima]


# --- TODO 2: Crear el agente ReAct con memoria ---
modelo = ChatOpenAI(
    model="google/gemma-4-31b-it:free",
    temperature=0,
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
)

agente = create_react_agent(
    modelo,
    tools=tools,
    checkpointer=MemorySaver(),
)


# Visualizar
print(agente.get_graph().draw_ascii())


# --- TODO 3: Conversación de prueba ---
config = {"configurable": {"thread_id": "alumno_1"}}

preguntas = [
    "¿Qué hora es y cuánto es la raíz cuadrada de 256?",
    "¿Qué tiempo hace en Madrid?",
    "¿Y en Bilbao?",  # debe entender que sigue hablando del clima
]

for p in preguntas:
    print("\n" + "=" * 60)
    print(f"Usuario: {p}")

    respuesta = agente.invoke(
        {"messages": [{"role": "user", "content": p}]},
        config=config,
    )

    # El último mensaje es la respuesta final del agente
    print(f"\nAgente: {respuesta['messages'][-1].content}")

    # TODO 4 (opcional): imprime los mensajes intermedios para ver el "razonamiento"
    print("\n--- Trazas internas ---")
    for m in respuesta["messages"]:
        m.pretty_print()

/tmp/ipykernel_3054/1571219037.py:71: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente = create_react_agent(


        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +-------+           
          | agent |           
          +-------+*          
          .         *         
        ..           **       
       .               *      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 

Usuario: ¿Qué hora es y cuánto es la raíz cuadrada de 256?


RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-4-31b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False}}, 'user_id': 'user_3CinYn5E7gvVmxVuMT6jMgdjrum'}

#### Verificación

Al ejecutar deberías observar:
1. La 1ª pregunta dispara dos llamadas a tools (hora_actual y calcular). Habilita el bloque de
"Trazas internas" para verlo.
2. La 2ª pregunta dispara clima("Madrid").
3. La 3ª pregunta solo dice "Bilbao": el agente debe inferir gracias a la memoria que el usuario sigue
preguntando por el tiempo.

#### Reto Adicional


* Añade una tool buscar_wikipedia(termino: str) que use wikipedia (pip install
wikipedia) para devolver el primer párrafo de un artículo. Pregunta al agente algo como "¿Quién
fue Ada Lovelace?" y observa cómo decide usar la nueva tool.

* Cambia ChatOpenAI por ChatAnthropic (Claude) y verifica que el mismo agente funciona sin
cambios adicionales.


##### Códigos

###### Código para ChatOpenAI

In [9]:
import os
import math
import wikipedia

from datetime import datetime
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

# Configurar Wikipedia en español
wikipedia.set_lang("es")


# --- TODO 1: Define 3 tools con el decorador @tool ---
# Cada tool DEBE tener un docstring claro: el LLM lo usará para decidir
# cuándo invocarla.
@tool
def calcular(expresion: str) -> str:
    """Evalúa una expresión matemática segura (suma, resta, mult, división,
    potencias, funciones de math). Ejemplo: 'sqrt(144) + 2**3'."""
    try:
        # eval restringido: solo names del módulo math + operadores
        permitidos = {
            k: getattr(math, k)
            for k in dir(math)
            if not k.startswith("_")
        }
        return str(eval(expresion, {"__builtins__": {}}, permitidos))
    except Exception as e:
        return f"Error: {e}"


# TODO: completa la siguiente tool. Debe devolver la fecha y hora actuales
# como string en formato "YYYY-MM-DD HH:MM".
@tool
def hora_actual() -> str:
    """Devuelve la fecha y hora actuales del sistema."""
    return datetime.now().strftime("%Y-%m-%d %H:%M")


# TODO: completa esta tool. Recibe un string `ciudad` y devuelve un string
# con el clima simulado (puedes hardcodear un dict de 3-4 ciudades).
@tool
def clima(ciudad: str) -> str:
    """Devuelve el clima actual de una ciudad española (datos simulados)."""
    datos_clima = {
        "madrid": "En Madrid hace sol, 18°C y viento suave.",
        "bilbao": "En Bilbao está nublado, 14°C y hay probabilidad de lluvia.",
        "barcelona": "En Barcelona hay intervalos nubosos, 20°C y humedad moderada.",
        "sevilla": "En Sevilla hace sol, 24°C y el ambiente es seco.",
    }

    ciudad_normalizada = ciudad.strip().lower()

    return datos_clima.get(
        ciudad_normalizada,
        f"No tengo datos simulados para {ciudad}. Prueba con Madrid, Bilbao, Barcelona o Sevilla."
    )


@tool
def buscar_wikipedia(termino: str) -> str:
    """Busca un término en Wikipedia y devuelve el primer párrafo del artículo más relevante."""
    try:
        resultados = wikipedia.search(termino)

        if not resultados:
            return f"No se encontraron resultados en Wikipedia para: {termino}"

        titulo = resultados[0]
        pagina = wikipedia.page(titulo, auto_suggest=False)

        parrafos = [
            p.strip()
            for p in pagina.content.split("\n\n")
            if p.strip()
        ]

        if not parrafos:
            return f"Se encontró el artículo '{titulo}', pero no tiene contenido disponible."

        return f"{pagina.title}: {parrafos[0]}"

    except wikipedia.exceptions.DisambiguationError as e:
        opciones = ", ".join(e.options[:5])
        return (
            f"El término '{termino}' es ambiguo. "
            f"Algunas opciones son: {opciones}."
        )

    except wikipedia.exceptions.PageError:
        return f"No se encontró una página de Wikipedia para: {termino}"

    except Exception as e:
        return f"Error consultando Wikipedia: {e}"


tools = [calcular, hora_actual, clima, buscar_wikipedia]


# --- TODO 2: Crear el agente ReAct con memoria ---
modelo = ChatOpenAI(
    model="google/gemma-4-31b-it:free",
    temperature=0,
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
)

agente = create_react_agent(
    modelo,
    tools=tools,
    checkpointer=MemorySaver(),
)


# Visualizar
print(agente.get_graph().draw_ascii())


# --- TODO 3: Conversación de prueba ---
config = {"configurable": {"thread_id": "alumno_1"}}

preguntas = [
    "¿Qué hora es y cuánto es la raíz cuadrada de 256?",
    "¿Qué tiempo hace en Madrid?",
    "¿Y en Bilbao?",  # debe entender que sigue hablando del clima
    "¿Quién fue Ada Lovelace?",
]

for p in preguntas:
    print("\n" + "=" * 60)
    print(f"Usuario: {p}")

    respuesta = agente.invoke(
        {"messages": [{"role": "user", "content": p}]},
        config=config,
    )

    # El último mensaje es la respuesta final del agente
    print(f"\nAgente: {respuesta['messages'][-1].content}")

    # TODO 4 (opcional): imprime los mensajes intermedios para ver el "razonamiento"
    print("\n--- Trazas internas ---")
    for m in respuesta["messages"]:
        m.pretty_print()

/tmp/ipykernel_3054/3793705678.py:114: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente = create_react_agent(


        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +-------+           
          | agent |           
          +-------+*          
          .         *         
        ..           **       
       .               *      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 

Usuario: ¿Qué hora es y cuánto es la raíz cuadrada de 256?


RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-4-31b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False}}, 'user_id': 'user_3CinYn5E7gvVmxVuMT6jMgdjrum'}

###### Código para anthropic
No tengo acceso a una api de claude, por lo que no voy a obtener resultados con la ejecución de este código.

In [6]:
!pip install -q wikipedia langchain-anthropic langgraph python-dotenv

from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.1/763.1 kB 20.2 MB/s eta 0:00:00


In [8]:
import os
import math
import wikipedia

from datetime import datetime
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain_anthropic import ChatAnthropic
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

# Configurar Wikipedia en español
wikipedia.set_lang("es")


# --- TODO 1: Define 3 tools con el decorador @tool ---
# Cada tool DEBE tener un docstring claro: el LLM lo usará para decidir
# cuándo invocarla.
@tool
def calcular(expresion: str) -> str:
    """Evalúa una expresión matemática segura (suma, resta, mult, división,
    potencias, funciones de math). Ejemplo: 'sqrt(144) + 2**3'."""
    try:
        # eval restringido: solo names del módulo math + operadores
        permitidos = {
            k: getattr(math, k)
            for k in dir(math)
            if not k.startswith("_")
        }

        return str(eval(expresion, {"__builtins__": {}}, permitidos))

    except Exception as e:
        return f"Error: {e}"


# TODO: completa la siguiente tool. Debe devolver la fecha y hora actuales
# como string en formato "YYYY-MM-DD HH:MM".
@tool
def hora_actual() -> str:
    """Devuelve la fecha y hora actuales del sistema."""
    return datetime.now().strftime("%Y-%m-%d %H:%M")


# TODO: completa esta tool. Recibe un string `ciudad` y devuelve un string
# con el clima simulado (puedes hardcodear un dict de 3-4 ciudades).
@tool
def clima(ciudad: str) -> str:
    """Devuelve el clima actual de una ciudad española usando datos simulados."""
    datos_clima = {
        "madrid": "En Madrid hace sol, 18°C y viento suave.",
        "bilbao": "En Bilbao está nublado, 14°C y hay probabilidad de lluvia.",
        "barcelona": "En Barcelona hay intervalos nubosos, 20°C y humedad moderada.",
        "sevilla": "En Sevilla hace sol, 24°C y el ambiente es seco.",
    }

    ciudad_normalizada = ciudad.strip().lower()

    return datos_clima.get(
        ciudad_normalizada,
        f"No tengo datos simulados para {ciudad}. Prueba con Madrid, Bilbao, Barcelona o Sevilla."
    )


# Tool adicional: buscar_wikipedia
@tool
def buscar_wikipedia(termino: str) -> str:
    """Busca un término en Wikipedia y devuelve el primer párrafo del artículo más relevante."""
    try:
        resultados = wikipedia.search(termino)

        if not resultados:
            return f"No se encontraron resultados en Wikipedia para: {termino}"

        titulo = resultados[0]
        pagina = wikipedia.page(titulo, auto_suggest=False)

        parrafos = [
            p.strip()
            for p in pagina.content.split("\n\n")
            if p.strip()
        ]

        if not parrafos:
            return f"Se encontró el artículo '{titulo}', pero no tiene contenido disponible."

        return f"{pagina.title}: {parrafos[0]}"

    except wikipedia.exceptions.DisambiguationError as e:
        opciones = ", ".join(e.options[:5])
        return (
            f"El término '{termino}' es ambiguo. "
            f"Algunas opciones son: {opciones}."
        )

    except wikipedia.exceptions.PageError:
        return f"No se encontró una página de Wikipedia para: {termino}"

    except Exception as e:
        return f"Error consultando Wikipedia: {e}"


tools = [calcular, hora_actual, clima, buscar_wikipedia]


# --- TODO 2: Crear el agente ReAct con memoria ---
modelo = ChatAnthropic(
    model="claude-3-5-haiku-latest",
    temperature=0,
    anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
)

agente = create_react_agent(
    modelo,
    tools=tools,
    checkpointer=MemorySaver(),
)


# Visualizar
print(agente.get_graph().draw_ascii())


# --- TODO 3: Conversación de prueba ---
config = {"configurable": {"thread_id": "alumno_1"}}

preguntas = [
    "¿Qué hora es y cuánto es la raíz cuadrada de 256?",
    "¿Qué tiempo hace en Madrid?",
    "¿Y en Bilbao?",  # debe entender que sigue hablando del clima
    "¿Quién fue Ada Lovelace?",
]

for p in preguntas:
    print("\n" + "=" * 60)
    print(f"Usuario: {p}")

    respuesta = agente.invoke(
        {"messages": [{"role": "user", "content": p}]},
        config=config,
    )

    # El último mensaje es la respuesta final del agente
    print(f"\nAgente: {respuesta['messages'][-1].content}")

    # TODO 4 (opcional): imprime los mensajes intermedios para ver el "razonamiento"
    print("\n--- Trazas internas ---")
    for m in respuesta["messages"]:
        m.pretty_print()

ValidationError: 1 validation error for ChatAnthropic
anthropic_api_key
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

#### Preguntas de Reflexión

##### 1. ¿Qué pasa si quitas el docstring de la tool clima? ¿Por qué la calidad de las decisiones del agente depende de los docstrings?


Si quitas el docstring de la tool clima, seguramente el agente tenga un problema para decidir cuándo debe usar esa herramienta.
Sin el docstring, el modelo tendría menos información contextual. Sabría que existe una herramienta "clima", pero sabe qué hace, cuándo usarla o qué tipo de argumento necesita.

La calidad de las decisiones del agente depende de los docstrings porque el LLM usa la descripción de cada herramienta para saber qué hace, cuándo invocarla y qué argumentos debe pasarle.

##### 2. Compara el código de este agente (~30 líneas) con el ejemplo de Function Calling manual de la teoría principal (sección 4.4). ¿Qué le añade create_react_agent que tú tendrías que escribir tú mismo?

El Function Calling manual permite llamar a las herramientas, pero hay que escribir manualmente la orquestación.
`create_react_agent` lo hace por si mismo: decide, invoca las herramientas, incorpora resultados, continúa el razonamiento y genera la respuesta final.

#####  3. ¿En qué situaciones reales sería útil este patrón (agente con tools)?  Da 3 ejemplos.

###### 1. Asistente para soporte técnico

Un agente podría tener tools para consultar tickets en Jira, revisar documentación interna, buscar errores conocidos, comprobar el estado de servicios y generar una respuesta para el cliente. Por ejemplo, ante una pregunta como “¿por qué está fallando el envío de correos?”, podría consultar logs, incidencias abiertas y documentación técnica antes de responder. Podría incluso trasladar el ticket al equipo responsable de la resolución de la incidencia.

##### 2. Asistente de análisis de datos

Un agente podría tener tools para ejecutar consultas SQL, cargar datasets, calcular métricas o generar gráficos. Por ejemplo, si el usuario pregunta “¿cuál fue la provincia con más alojamientos turísticos?”, el agente podría consultar una base de datos o un CSV, calcular el resultado y explicarlo.



##### 3. Asistente personal o de productividad

Un agente podría tener tools para consultar el calendario, crear recordatorios, enviar correos o buscar información actualizada. Por ejemplo, si el usuario pregunta “organízame una reunión mañana con Juan y mándale un resumen”, el agente podría mirar disponibilidad, crear el evento y redactar el correo.